# FL-PUB-005.5 — IEEE/ACM manuscript draft

*Epic EPIC-001 · story fl_pub_005 · all results below are loaded from files produced by the task's experiment function and labelled MEASURED/SIMULATED; clients are virtual; data is synthetic.*

## 1. Research objective
Produce an IEEE-style manuscript draft and verified-status bibliography.

## 2. Scientific background
—

References use keys of `docs/references/references_master.md` (status per entry).

## 3. Research questions
- **RQ1.** Does the document state its assumptions, limits and sources honestly and consistently with the stored results?

## 4. Hypotheses
- **H1.** Every quantitative statement is traceable to a stored result file; no compliance claim appears.

## 5. Mathematical formulation
—

## 6. Experimental environment

In [1]:
import os, sys, json, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
def _find_root():
    p = Path(os.environ.get("FL_ROOT") or os.environ.get("RS_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set RS_ROOT")
ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src"))
STORY = "fl_pub_005"
_cands = [ROOT / "experiments" / "results" / STORY]
RES = next(c for c in _cands if c.exists())
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
SEED = 0; np.random.seed(SEED)
def show_csv(name, n=12):
    p = RES / name
    if p.exists(): display(pd.read_csv(p).round(4).head(n))
    else: print("MISSING", name)
def show_fig(name):
    for base in (RES / "figures", RES):
        if (base / name).exists(): display(Image(filename=str(base / name))); return
    print("MISSING figure", name)
print("root:", ROOT, "| results:", RES)

root: /home/claude/fl_sprint | results: /home/claude/fl_sprint/experiments/results/fl_pub_005


In [2]:
import importlib.metadata as md
vers = {}
for p in ["numpy","scipy","scikit-learn","torch","tenseal","phe","pandas","matplotlib","cryptography","psutil","mlflow","opacus"]:
    try: vers[p] = md.version(p)
    except Exception: vers[p] = "not installed"
print(json.dumps(dict(python=platform.python_version(), platform=platform.platform(), versions=vers, seed=SEED, gpu="none (CPU only; GPU metrics NOT_MEASURED)"), indent=1))

{
 "python": "3.13.18",
 "platform": "Linux-6.18.44-fc-v338-x86_64-with-glibc2.39",
 "versions": {
  "numpy": "2.5.3",
  "scipy": "1.18.1",
  "scikit-learn": "1.9.1",
  "torch": "2.15.0",
  "tenseal": "0.3.18",
  "phe": "1.5.0",
  "pandas": "3.0.5",
  "matplotlib": "3.11.2",
  "cryptography": "50.0.1",
  "psutil": "7.2.2",
  "mlflow": "3.18.0",
  "opacus": "1.6.0"
 },
 "seed": 0,
 "gpu": "none (CPU only; GPU metrics NOT_MEASURED)"
}


## 7. Dataset description
Synthetic binary tabular dataset **SYN-IDS-v1** (scikit-learn `make_classification`, 40 features, 70:30 split). Chosen because no real IDS/clinical dataset was available in the sandbox. It has no clinical meaning; conclusions about absolute accuracy do not transfer.

In [3]:
from federated_learning.synthetic_data.generators.syn_ids_v1 import make_dataset, DATASET_ID
from federated_learning.data_partitioning.iid import partition_iid
from federated_learning.data_partitioning.dirichlet import partition_dirichlet
from federated_learning.data_partitioning.metrics import label_skew_stat
data = make_dataset(n_samples=6000, seed=SEED)
print(DATASET_ID, "train", data.X_train.shape, "test", data.X_test.shape, "positive rate", float(data.y_train.mean()))
print("NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.")

SYN-IDS-v1 train (4200, 40) test (1800, 40) positive rate 0.40095238095238095
NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.


## 8. Configuration and reproducibility settings
Seeds are fixed; each stored result carries `provenance` (experiment id, SHA-256 of the configuration, environment versions; `git_commit` is `null` because the workspace is not a git repository).

In [4]:
print("No FL configuration is needed to read stored results; see the stored JSON `provenance` blocks.")
print(json.dumps(json.load(open(next(RES.glob("*.json"))))["provenance"]["config"], indent=1)[:800])

No FL configuration is needed to read stored results; see the stored JSON `provenance` blocks.
{
 "K": 20,
 "rounds": 20,
 "seeds": 3,
 "sigma": 0.5,
 "target": 0.85,
 "byz_frac": 0.2,
 "backdoor": "medium"
}


## 9. Data preparation
Partitioning into virtual clients (IID or Dirichlet label skew) is performed by `federated_learning.data_partitioning`.

In [5]:
parts_iid = partition_iid(data.y_train, 10, SEED)
parts_dir = partition_dirichlet(data.y_train, 10, 0.3, SEED)
print('label-skew statistic IID:', round(label_skew_stat(data.y_train, parts_iid),3), ' Dirichlet(0.3):', round(label_skew_stat(data.y_train, parts_dir),3))

label-skew statistic IID: 0.019  Dirichlet(0.3): 0.363


## 10. Implementation
Reusable code lives in `src/federated_learning/` (`scripts/generate_publication.py`). The cell below exercises it live (small, fast, deterministic) and asserts correctness.

In [6]:
import re
P = ROOT / 'latex/manuscript_ieee.tex'
txt = P.read_text() if P.exists() else ''
print(P.name, 'exists:', P.exists(), '| characters:', len(txt))
banned = [w for w in ['is compliant', 'GDPR-compliant', 'guarantees compliance', 'certified'] if w in txt]
print('compliance-claim phrases found:', banned); assert not banned

manuscript_ieee.tex exists: True | characters: 33438
compliance-claim phrases found: []


## 11. Experiment execution
The full experiment was executed by `python -m federated_learning.pipelines.experiments.run_all --task FL-PUB-005.5` (see `logs/`). Its raw per-run results are loaded here; they are not re-computed inside the notebook.

In [7]:
show_csv('task_5_2_incompatible.csv')

,combination,reason,status
0,SecAgg + median/Krum/trimmed-mean,robust rules need per-client updates; masked u...,NOT_RUN (incompatible by construction)
1,HE (CKKS) + Krum/median,needs encrypted comparisons/sorting (deep circ...,NOT_RUN
2,plain + DP + robust (median),composition would need a separate privacy anal...,NOT_RUN


## 12. Metrics collection
Metric definitions: document checks: existence, absence of compliance claims, presence of limitations section.

In [8]:
show_csv('task_5_2_summary.csv')

,config,acc,acc_ci,f1,f1_ci,rounds_to_target,rounds_to_target_ci,train_overhead_pct,train_overhead_pct_ci,uplink_overhead_pct,uplink_overhead_pct_ci,acc_loss_pp,acc_loss_pp_ci,acc_under_signflip,acc_under_signflip_ci,backdoor_asr,backdoor_asr_ci,server_sees_individual_updates,gradient_leakage_success_B2,gl_source,mia_auc_undefended_model
0,plain FL,0.8477,"[0.836, 0.859]",0.8029,"[0.786, 0.82]",20.6667,"[19.2, 22.1]",0.0000,"[0, 0]",0.0000,"[0, 0]",0.0000,"[0, 0]",0.4156,"[0.0331, 0.798]",0.9972,"[0.996, 0.999]",True,0.8,task 3.4 (measured there): single client B=2,0.5772304999999999
1,+HE (CKKS),0.8477,"[0.836, 0.859]",0.8029,"[0.786, 0.82]",20.6667,"[19.2, 22.1]",60.2495,"[55.4, 65.1]",5912.9170,"[5.91e+03, 5.92e+03]",0.0000,"[0, 0]",0.4156,"[0.0331, 0.798]",0.9972,"[0.996, 0.999]",False,0.0,task 3.4 (measured there): secure-aggregation ...,0.5772304999999999
2,+SecAgg,0.8477,"[0.836, 0.859]",0.8029,"[0.786, 0.82]",20.6667,"[19.2, 22.1]",29.8722,"[26.1, 33.6]",183.8897,"[184, 184]",0.0000,"[0, 0]",0.4156,"[0.0331, 0.798]",0.9972,"[0.996, 0.999]",False,0.0,task 3.4 (measured there): secure-aggregation ...,0.5772304999999999
3,+SMPC (3-party),0.8477,"[0.836, 0.859]",0.8029,"[0.786, 0.82]",20.6667,"[19.2, 22.1]",92.7490,"[66.6, 119]",500.0000,"[500, 500]",0.0000,"[0, 0]",0.4156,"[0.034, 0.797]",0.9972,"[0.996, 0.999]",False,0.0,task 3.4 (measured there): secure-aggregation ...,0.5772304999999999
4,"+DP (central, sigma=0.5)",0.8211,"[0.814, 0.828]",0.7656,"[0.755, 0.776]",21.0000,"[21, 21]",-2.5362,"[-6.45, 1.38]",0.0000,"[0, 0]",2.6574,"[0.922, 4.39]",0.7762,"[0.768, 0.784]",0.8550,"[0.6, 1.11]",True,0.8,task 3.4 (measured there): single client B=2,see task 3.3 (DP configs)
5,+robust (median),0.8460,"[0.84, 0.852]",0.8006,"[0.792, 0.81]",21.0000,"[21, 21]",-5.6885,"[-8.6, -2.78]",0.0000,"[0, 0]",0.1667,"[-0.447, 0.78]",0.8189,"[0.817, 0.821]",0.1335,"[-0.161, 0.428]",True,0.8,task 3.4 (measured there): single client B=2,0.5772304999999999
6,SecAgg + client-DP,0.8225,"[0.811, 0.834]",0.7684,"[0.752, 0.785]",21.0000,"[21, 21]",38.6802,"[34.2, 43.2]",183.8897,"[184, 184]",2.5185,"[0.269, 4.77]",0.7806,"[0.762, 0.799]",0.8570,"[0.633, 1.08]",False,0.0,task 3.4 (measured there): secure-aggregation ...,see task 3.3 (DP configs)
7,HE + client-DP,0.8225,"[0.811, 0.834]",0.7684,"[0.752, 0.785]",21.0000,"[21, 21]",70.9800,"[42.3, 99.6]",5910.5848,"[5.91e+03, 5.91e+03]",2.5185,"[0.269, 4.77]",0.7806,"[0.762, 0.799]",0.8570,"[0.633, 1.08]",False,0.0,task 3.4 (measured there): secure-aggregation ...,see task 3.3 (DP configs)
8,SMPC + norm-bound,0.8478,"[0.836, 0.859]",0.8030,"[0.786, 0.82]",20.6667,"[19.2, 22.1]",394.7630,"[381, 409]",500.0000,"[500, 500]",-0.0093,"[-0.0491, 0.0306]",0.8442,"[0.835, 0.854]",0.0458,"[-0.0948, 0.186]",False,0.0,task 3.4 (measured there): secure-aggregation ...,0.5772304999999999


## 13. Statistical analysis
Not applicable.

In [9]:
print('n/a')

n/a


## 14. Visualizations

In [10]:
print('no figure for this task (tabular result)')

no figure for this task (tabular result)


## 15. Interpretation of results
The numbers are computed in the next cell from the stored data; the text interprets only what that cell prints.

Document check only; IEEEtran.cls unavailable in the sandbox, compile-checked with the article class.

In [11]:
t = (ROOT / 'latex/manuscript_ieee.tex').read_text() if (ROOT / 'latex/manuscript_ieee.tex').exists() else ''
print({'mentions_limitations': bool(re.search('(?i)limitation|threats to validity', t)), 'mentions_virtual_clients': 'virtual' in t.lower(), 'mentions_not_verified': 'NOT_VERIFIED' in t})

{'mentions_limitations': True, 'mentions_virtual_clients': True, 'mentions_not_verified': False}


## 16. Limitations
- Documents are drafts; legal sources unverified.
- Synthetic data, virtual clients on one machine, 2 CPU cores, no GPU; no real network. Comm. bytes are logical payload sizes.
- No regulatory/compliance conclusion can be drawn from this notebook.

## 17. Conclusions
Draft produced; see file.

**Jira acceptance:** see `docs/fl_pub_005/results_and_discussion.md` for the criterion-by-criterion verdict (MET / PARTIALLY MET / NOT MET).

## 18. References
`gdpr`, `aiact`, `nis2`, `kairouz2021` — details in `docs/fl_pub_005/references.md`.